# Stage 02 · Step 0 — Generate multi-τ training corpus

The supervised RUL head benefits from telemetry observed under several maintenance schedules, not just the single baseline τ already in `data/fleet_baseline.parquet`. This notebook draws K τ vectors via Latin-Hypercube sampling over the same ranges Stage 01 explores and runs the SDG simulator on the **train printers (id 0..69)** for each.

Outputs:
- `data/policy_runs/policy_{k:03d}.parquet` — one parquet per τ schedule (train printers only, RUL labels included).
- `data/policy_runs/manifest.json` — index of τ values per file.

Skip this notebook if the corpus already exists; the SSL pretraining notebook will fall back to `fleet_baseline.parquet` if no policy runs are found.

In [1]:
from __future__ import annotations
import json
from pathlib import Path

import numpy as np
import pyarrow.parquet as pq
from scipy.stats import qmc

from ml.lib.data import TRAIN_PRINTERS
from ml.lib.env_runner import default_dates, run_with_tau
from ml import PROJECT_ROOT
from backend.simulator.generate import load_configs
from backend.simulator.labels import add_rul_labels
from backend.simulator.schema import COMPONENT_IDS, table_from_dataframe

POLICY_DIR = PROJECT_ROOT / 'data/policy_runs'
POLICY_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = POLICY_DIR / 'manifest.json'

In [2]:
TAU_RANGES = {
    'C1': (50.0, 2_000.0),
    'C2': (500.0, 20_000.0),
    'C3': (24.0, 500.0),
    'C4': (100.0, 2_000.0),
    'C5': (500.0, 8_000.0),
    'C6': (1_000.0, 20_000.0),
}
# K = number of LHS-sampled τ schedules to simulate. The SSL pretraining corpus
# grows linearly with K (each run = ~21 MB on disk, ~70 printers × 10 years of
# rows). 5 = smoke run; 30 = surrogate-quality default; 60+ = recommended for
# Stage 03 RL+SSL because the policy will see a much richer τ distribution
# at SSL-pretrain time → better embeddings → better RL finetuning.
# Wall-clock: ~2–5 min per run on CPU, so K=60 ≈ 2–5 h. Cells below are
# resumable — re-running skips runs already on disk.
K = 60
SEED = 17

sampler = qmc.LatinHypercube(d=len(TAU_RANGES), seed=SEED)
unit = sampler.random(K)
tau_schedules: list[dict[str, float]] = []
for row in unit:
    schedule = {}
    for i, (component_id, (low, high)) in enumerate(TAU_RANGES.items()):
        schedule[component_id] = float(np.exp(np.log(low) + row[i] * (np.log(high) - np.log(low))))
    tau_schedules.append(schedule)
tau_schedules

[{'C1': 1679.050096491176,
  'C2': 1592.1752150934856,
  'C3': 397.00311253193183,
  'C4': 723.3890763792543,
  'C5': 823.0225525011688,
  'C6': 1973.36545600051},
 {'C1': 122.47644779008843,
  'C2': 18113.6313022198,
  'C3': 262.4417931789659,
  'C4': 173.05758046489672,
  'C5': 7549.584636891962,
  'C6': 4561.404987796921},
 {'C1': 699.6440492676493,
  'C2': 4863.699235775497,
  'C3': 128.5918599854882,
  'C4': 110.29864885814096,
  'C5': 1285.3524541642682,
  'C6': 1071.9013961519088},
 {'C1': 845.3392828769373,
  'C2': 3325.9773042654992,
  'C3': 43.685581131863664,
  'C4': 684.9399202978318,
  'C5': 1860.2971592319127,
  'C6': 2401.6508729744737},
 {'C1': 961.0811667696258,
  'C2': 12369.82927183978,
  'C3': 112.91524510113781,
  'C4': 465.3647779677169,
  'C5': 1183.2691718369542,
  'C6': 8198.088574253168},
 {'C1': 538.2639258638743,
  'C2': 680.3205043029915,
  'C3': 57.996750379146846,
  'C4': 539.6530114124972,
  'C5': 844.4517195977671,
  'C6': 10629.375633895743},
 {'C1': 1

In [3]:
components_cfg, couplings_cfg, cities_cfg = load_configs()
DATES = default_dates()
manifest = []
for k, schedule in enumerate(tau_schedules):
    output_path = POLICY_DIR / f'policy_{k:03d}.parquet'
    if output_path.exists():
        print(f'[{k}] cached:', output_path)
        manifest.append({'k': k, 'tau': schedule, 'path': output_path.as_posix()})
        continue
    print(f'[{k}] simulating', schedule)
    df = run_with_tau(
        schedule,
        printer_ids=TRAIN_PRINTERS,
        dates=DATES,
        components_cfg=components_cfg,
        couplings_cfg=couplings_cfg,
        cities_cfg=cities_cfg,
    )
    table = table_from_dataframe(df, include_rul=False)
    pq.write_table(table, output_path, compression='snappy', version='2.6')
    add_rul_labels(output_path)
    manifest.append({'k': k, 'tau': schedule, 'path': output_path.as_posix()})
    print(f'[{k}] wrote {output_path} ({output_path.stat().st_size / 1e6:.1f} MB)')

[0] cached: C:\Users\Sergi\OneDrive\Desktop\Industrial Engineering\Q8\TFG\Coding\hackupc2026\data\policy_runs\policy_000.parquet
[1] cached: C:\Users\Sergi\OneDrive\Desktop\Industrial Engineering\Q8\TFG\Coding\hackupc2026\data\policy_runs\policy_001.parquet
[2] cached: C:\Users\Sergi\OneDrive\Desktop\Industrial Engineering\Q8\TFG\Coding\hackupc2026\data\policy_runs\policy_002.parquet
[3] cached: C:\Users\Sergi\OneDrive\Desktop\Industrial Engineering\Q8\TFG\Coding\hackupc2026\data\policy_runs\policy_003.parquet
[4] cached: C:\Users\Sergi\OneDrive\Desktop\Industrial Engineering\Q8\TFG\Coding\hackupc2026\data\policy_runs\policy_004.parquet
[5] cached: C:\Users\Sergi\OneDrive\Desktop\Industrial Engineering\Q8\TFG\Coding\hackupc2026\data\policy_runs\policy_005.parquet
[6] cached: C:\Users\Sergi\OneDrive\Desktop\Industrial Engineering\Q8\TFG\Coding\hackupc2026\data\policy_runs\policy_006.parquet
[7] cached: C:\Users\Sergi\OneDrive\Desktop\Industrial Engineering\Q8\TFG\Coding\hackupc2026\data

[20] wrote C:\Users\Sergi\OneDrive\Desktop\Industrial Engineering\Q8\TFG\Coding\hackupc2026\data\policy_runs\policy_020.parquet (39.1 MB)
[21] simulating {'C1': 376.2626374391061, 'C2': 1932.5633900610471, 'C3': 35.24513748065179, 'C4': 1310.574072312171, 'C5': 2977.9678329518656, 'C6': 1722.9788898227885}


[21] wrote C:\Users\Sergi\OneDrive\Desktop\Industrial Engineering\Q8\TFG\Coding\hackupc2026\data\policy_runs\policy_021.parquet (39.0 MB)
[22] simulating {'C1': 278.89355405957315, 'C2': 4607.587714064778, 'C3': 87.71676630176154, 'C4': 183.26384846046585, 'C5': 7018.376958556121, 'C6': 4801.999112891984}


[22] wrote C:\Users\Sergi\OneDrive\Desktop\Industrial Engineering\Q8\TFG\Coding\hackupc2026\data\policy_runs\policy_022.parquet (39.1 MB)
[23] simulating {'C1': 299.8638344451452, 'C2': 1048.7401510783302, 'C3': 250.74720649281116, 'C4': 404.65651998069836, 'C5': 4892.6016340709475, 'C6': 1454.6993543663332}


[23] wrote C:\Users\Sergi\OneDrive\Desktop\Industrial Engineering\Q8\TFG\Coding\hackupc2026\data\policy_runs\policy_023.parquet (38.7 MB)
[24] simulating {'C1': 1164.9547594814137, 'C2': 14385.86152856225, 'C3': 78.06803669265479, 'C4': 119.89980471835406, 'C5': 1118.1132617735059, 'C6': 7203.757607811914}


...

[55] wrote C:\Users\Sergi\OneDrive\Desktop\Industrial Engineering\Q8\TFG\Coding\hackupc2026\data\policy_runs\policy_055.parquet (38.9 MB)
[56] simulating {'C1': 222.37183258129866, 'C2': 2362.932344507908, 'C3': 155.54965168089518, 'C4': 1038.7466549565147, 'C5': 3797.33853974824, 'C6': 1140.9463153394997}


[56] wrote C:\Users\Sergi\OneDrive\Desktop\Industrial Engineering\Q8\TFG\Coding\hackupc2026\data\policy_runs\policy_056.parquet (39.1 MB)
[57] simulating {'C1': 85.42271855584201, 'C2': 667.0922252541352, 'C3': 39.260054501314166, 'C4': 147.36705905293078, 'C5': 1067.5684448927086, 'C6': 3978.5518208410704}


[57] wrote C:\Users\Sergi\OneDrive\Desktop\Industrial Engineering\Q8\TFG\Coding\hackupc2026\data\policy_runs\policy_057.parquet (39.0 MB)
[58] simulating {'C1': 95.98585777623785, 'C2': 798.9234147960566, 'C3': 304.9298244726005, 'C4': 874.6250753451069, 'C5': 702.7537201910635, 'C6': 17543.44436618907}


[58] wrote C:\Users\Sergi\OneDrive\Desktop\Industrial Engineering\Q8\TFG\Coding\hackupc2026\data\policy_runs\policy_058.parquet (39.2 MB)
[59] simulating {'C1': 52.30920461142501, 'C2': 1372.5620455854332, 'C3': 70.14655423729918, 'C4': 758.907936057186, 'C5': 680.2206924413357, 'C6': 1319.418700388474}


[59] wrote C:\Users\Sergi\OneDrive\Desktop\Industrial Engineering\Q8\TFG\Coding\hackupc2026\data\policy_runs\policy_059.parquet (38.7 MB)


In [4]:
with MANIFEST_PATH.open('w', encoding='utf-8') as handle:
    json.dump({'tau_ranges': TAU_RANGES, 'seed': SEED, 'runs': manifest}, handle, indent=2)
print('Wrote manifest:', MANIFEST_PATH)
print(f'{len(manifest)} policy runs covering printer_ids {TRAIN_PRINTERS[0]}..{TRAIN_PRINTERS[-1]}')

Wrote manifest: C:\Users\Sergi\OneDrive\Desktop\Industrial Engineering\Q8\TFG\Coding\hackupc2026\data\policy_runs\manifest.json
60 policy runs covering printer_ids 0..69
